## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-04-04-53-35 +0000,bbc,What the Democratic takeover of the Senate mea...,https://www.bbc.co.uk/news/videos/cr5ynex46j75...
1,2026-11-04-04-53-30 +0000,startribune,Democrat David Crowley defeats Trump loyalist ...,https://www.startribune.com/crowley-looks-to-m...
2,2026-11-04-04-51-56 +0000,nyt,"Luria Wins in Virginia, Flipping a Key House S...",https://www.nytimes.com/2026/11/03/us/politics...
3,2026-11-04-04-50-47 +0000,chitri,Democrat Barbara Domer in lead for Porter Coun...,https://www.chicagotribune.com/2026/11/03/demo...
4,2026-11-04-04-50-29 +0000,startribune,Star Tribune Opinion reacts to the 2026 midter...,https://www.startribune.com/klobuchar-demuth-f...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2438/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
87,election,47
2,senate,42
65,race,39
4,trump,37
22,democrats,34


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
101,2026-11-04-03-14-56 +0000,chitri,South Carolina Senate race: Democrat Annie And...,https://www.chicagotribune.com/2026/11/03/sout...,245
162,2026-11-04-00-55-30 +0000,chitri,North Carolina Senate race: Democrat Roy Coope...,https://www.chicagotribune.com/2026/11/03/nort...,237
208,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...,232
149,2026-11-04-01-04-38 +0000,chitri,Georgia Senate race: Sen. Jon Ossoff wins reel...,https://www.chicagotribune.com/2026/11/03/geor...,217
25,2026-11-04-04-38-02 +0000,startribune,Democrat Roy Cooper wins North Carolina Senate...,https://www.startribune.com/roy-cooper-looks-t...,217


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
101,245,2026-11-04-03-14-56 +0000,chitri,South Carolina Senate race: Democrat Annie And...,https://www.chicagotribune.com/2026/11/03/sout...
208,232,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...
24,111,2026-11-04-04-38-04 +0000,startribune,"Georgia Sen. Jon Ossoff wins reelection, defea...",https://www.startribune.com/sen-jon-ossoff-tri...
132,88,2026-11-04-02-06-47 +0000,nypost,New York Gov. Hochul defeats GOP candidate Bru...,https://nypost.com/2026/11/03/us-news/gov-hoch...
401,78,2026-11-03-11-57-00 +0000,wsj,"Follow the WSJ’s Election Day live coverage, a...",https://www.wsj.com/politics/elections/your-ho...
120,75,2026-11-04-02-35-53 +0000,chitri,Georgia governor race: Democrat Keisha Lance B...,https://www.chicagotribune.com/2026/11/03/geor...
33,75,2026-11-04-04-33-12 +0000,chitri,Lake County Sheriff John Idleburg declares vic...,https://www.chicagotribune.com/2026/11/03/lake...
203,72,2026-11-03-23-00-00 +0000,wsj,Democrats and Republicans are battling over th...,https://www.wsj.com/politics/elections/early-e...
189,67,2026-11-03-23-39-55 +0000,nyt,OpenAI and Anthropic Set Up ‘War Rooms’ to Fig...,https://www.nytimes.com/2026/11/03/business/ai...
155,54,2026-11-04-01-00-20 +0000,chitri,Voters usher in new era with state Rep. La Sha...,https://www.chicagotribune.com/2026/11/03/illi...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
